### Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2
import kagglehub


### Step 1: Download ORL Dataset

In [ ]:
# Download latest version
path = kagglehub.dataset_download("kasikrit/att-database-of-faces")
print("Path to dataset files:", path)


### Step 2: Generate the Data Matrix and the Label vector

In the AT&T Face Dataset, a subject refers to a person. The dataset includes 40 different people, and each one is stored in a separate folder
Reading all 400 images (10 per person × 40 people)
Storing them in images (shape: 400, 112, 92)
Assigning a label (person ID) from 1 to 40 to each image

In [ ]:
images = [] # data matrix
Y = []  # label vector

# Loop through subjects (s1 to s40)
for subject_id in range(1, 41): 
    subject_folder = os.path.join(path, f's{subject_id}')
    
    # Loop through each of the 10 images for the subject
    for img_number in range(1, 11):  # 1.pgm to 10.pgm
        img_path = os.path.join(subject_folder, f'{img_number}.pgm')
        
        # Read image 
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        # Append image and label
        images.append(img)
        Y.append(subject_id)  # Use subject_id as the label

images = np.array(images)        
Y = np.array(Y) 
D = images.reshape(400, -1)
D.shape

print("Loaded dataset shape:", D.shape)
print("Y shape:", Y.shape)


### Step 3: Split the Dataset into Training and Test sets

In [ ]:
# Select training and testing rows
D_train = D[::2]  # odd-numbered rows
D_test  = D[1::2] # even-numbered rows 

y_train = Y[::2]
y_test  = Y[1::2]

print("Training set shape:", D_train.shape)
print("Testing set shape:", D_test.shape)
print("Training labels shape:", y_train.shape)
print("Testing labels shape:", y_test.shape)

### PCA Implementation

In [ ]:
class PCA:
    def __init__(self, D_train, D_test, y_train, y_test):
        self.D_train = D_train
        self.D_test = D_test
        self.y_train = y_train
        self.y_test = y_test
        self.D_centered = []
        self.eigenvalues = []
        self.eigenvectors = []

    def compCov(self):
        mean_face = np.mean(self.D_train, axis=0)
        self.D_centered = self.D_train - mean_face
        Cov = self.D_centered.T @ self.D_centered / self.D_train.shape[0]
        return Cov

    def compEig(self, Cov, save_to_file=True):
        eigenvalues, eigenvectors = np.linalg.eigh(Cov)
        eigenvalues = eigenvalues[::-1]
        eigenvectors = eigenvectors[:, ::-1]
        norm_eigenvect = eigenvectors / np.linalg.norm(eigenvectors, axis=0)

        self.eigenvalues = eigenvalues
        self.eigenvectors = norm_eigenvect

        if save_to_file:
            np.save("eigenvalues.npy", self.eigenvalues)
            np.save("eigenvectors.npy", self.eigenvectors)

    def loadEig(self):
        """Load eigenvalues and eigenvectors from .npy files"""
        self.eigenvalues = np.load("eigenvalues.npy")
        self.eigenvectors = np.load("eigenvectors.npy")

    def selectComp(self, alpha):
        sum_var = 0
        eigsum = np.sum(self.eigenvalues)
        for i, val in enumerate(self.eigenvalues):
            sum_var += val
            if sum_var / eigsum >= alpha:
                return self.eigenvectors[:, :i+1]

    def project(self, alpha, use_saved_eig=False):
        if use_saved_eig:
            self.loadEig()
            mean_face = np.mean(self.D_train, axis=0)
            self.D_centered = self.D_train - mean_face
        else:
            Cov = self.compCov()
            self.compEig(Cov)
        
        eigvec = self.selectComp(alpha)
        proj = self.D_centered @ eigvec
        return proj, eigvec

    def visualize_projection(self, alpha, x, image_shape=(112, 92), use_saved_eig=False):
        projected, subeigvec = self.project(alpha, use_saved_eig=use_saved_eig)
        recon = projected[x] @ subeigvec.T
        mean_face = np.mean(self.D_train, axis=0)
        recon_image = recon + mean_face
        recon_image = recon_image.reshape(image_shape)
        plt.imshow(recon_image, cmap='gray')
        plt.title(f"Reconstructed Image with alpha={alpha} and Dimensions ={projected.shape[1]}")
        plt.axis('off')
        plt.show()

pca = PCA(D_train, D_test, y_train, y_test)
#Cov = pca.compCov()
#pca.compEig(Cov, save_to_file=True)




In [ ]:
plt.imshow(images[2], cmap='gray')
plt.title("Original image")
plt.axis('off')
plt.show()

pca.visualize_projection(alpha=0.95, x=1, use_saved_eig=True)
pca.visualize_projection(alpha=0.9, x=1, use_saved_eig=True)
pca.visualize_projection(alpha=0.85, x=1, use_saved_eig=True)
pca.visualize_projection(alpha=0.8, x=1, use_saved_eig=True)
pca.visualize_projection(alpha=0.5, x=1, use_saved_eig=True)

## Unsupervised Clustering
### K-Means Clustering

In [ ]:
class CustomKMeans:
    def __init__(self, n_clusters, max_iter=100):
        """Class initialization"""
        self.n_clusters = n_clusters
        self.max_iter = max_iter

    def fit(self, X):
        """Fit the KMeans model to the data"""
        # Randomly initialize centroids
        np.random.seed(42)
        indices = np.random.choice(X.shape[0], self.n_clusters, replace=False)  # Randomly select initial centroids
        self.centroids = X[indices]
        
        for _ in range(self.max_iter):
            # Compute distances from each point to each centroid
            distances = np.linalg.norm(X[:, np.newaxis] - self.centroids, axis=2)
            self.labels_ = np.argmin(distances, axis=1) + 1  # Shift labels to start from 1
            
            new_centroids = np.array([
                X[self.labels_ == i].mean(axis=0) if np.any(self.labels_ == i) else self.centroids[i - 1]
                for i in range(1, self.n_clusters + 1)
            ])
            
            # Check for convergence
            if np.allclose(self.centroids, new_centroids):
                break

            self.centroids = new_centroids
            
            # print("Updated centroids:\n", self.centroids)
            # print("Updated labels:\n", self.labels_)
            # print("Updated centroids shape:", self.centroids.shape)
            # print("Updated labels shape:", self.labels_.shape)
            

    def predict(self, X):
        """Predict the closest cluster each sample in X belongs to"""
        distances = np.linalg.norm(X[:, np.newaxis] - self.centroids, axis=2)
        return np.argmin(distances, axis=1) + 1  # Shift labels to start from 1


### K-Means Clustering Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score
from scipy.optimize import linear_sum_assignment

def clustering_accuracy(y_true, y_pred):
    """Compute clustering accuracy using the Hungarian algorithm"""
    D = max(y_pred.max(), y_true.max()) + 1 # Number of clusters
    # print("Number of clusters:", D)
    # Initialize the cost matrix
    cost = np.zeros((D, D), dtype=int)
    
    # Count the number of points assigned to each cluster
    for i in range(len(y_pred)):
        cost[y_pred[i], y_true[i]] += 1
        
    row_ind, col_ind = linear_sum_assignment(cost.max() - cost)
    total_correct = sum(cost[i, j] for i, j in zip(row_ind, col_ind))   # 
    
    # print("Cost matrix:\n", cost)
    # print("Cost matrix shape:", cost.shape)
    # print("Row indices:", row_ind)
    # print("Column indices:", col_ind)
    # print("Total correct assignments:", total_correct)
    # print("---------------------------------------------------------")
    return total_correct / len(y_pred)


### Run K-Means for different alpha and K

In [ ]:
results_kmeans = []

for alpha in [0.8, 0.85, 0.9, 0.95]:
    reduced_data, eigvecs = pca.project(alpha,use_saved_eig=True)

    for K in [20, 40, 60]:
        kmeans = CustomKMeans(n_clusters=K)
        kmeans.fit(reduced_data)

        acc = clustering_accuracy(pca.y_train, kmeans.labels_)
        results_kmeans.append((alpha, K, acc))
        print(f"Alpha: {alpha}, K: {K}, Accuracy: {acc:.4f}")
    print("--------------------------------------")



###  Plot Accuracy vs K and Alpha (K-Means)

In [ ]:
import matplotlib.pyplot as plt

for alpha in [0.8, 0.85, 0.9, 0.95]:
    accs = [acc for a, k, acc in results_kmeans if a == alpha]
    ks = [k for a, k, acc in results_kmeans if a == alpha]
    plt.plot(ks, accs, marker='o', label=f'α={alpha}')

plt.xlabel('K')
plt.ylabel('Clustering Accuracy')
plt.title('Accuracy vs K for Different α')
plt.legend()
plt.grid()
plt.show()

for k in [20, 40, 60]:
    accs = [acc for a, k_val, acc in results_kmeans if k_val == k]
    alphas = [a for a, k_val, acc in results_kmeans if k_val == k]
    plt.plot(alphas, accs, marker='o', label=f'K={k}')
    

plt.xlabel('α')
plt.ylabel('Clustering Accuracy')
plt.title('Accuracy vs α for Different K')  
plt.legend()
plt.grid()
plt.show()


### Evaluate Best Model on Test Set

In [ ]:
# Find the entry with the highest accuracy
best_alpha, best_K, best_acc = max(results_kmeans, key=lambda x: x[2])

print(f"Best alpha: {best_alpha}, Best K: {best_K}, Accuracy: {best_acc:.4f}")

# Get the PCA eigenvectors for best_alpha
_, eigvecs = pca.project(best_alpha,use_saved_eig=True)
mean_face = np.mean(pca.D_train, axis=0)
D_test_centered = pca.D_test - mean_face
test_proj = D_test_centered @ eigvecs

# Fit KMeans on train set (projected)
train_proj, _ = pca.project(best_alpha,use_saved_eig=True)
kmeans_best = CustomKMeans(n_clusters=best_K)
kmeans_best.fit(train_proj)

# Predict on test set
y_pred_test = kmeans_best.predict(test_proj)

# Evaluate clustering accuracy
acc = clustering_accuracy(pca.y_test, y_pred_test)
print(f"Test Accuracy: {acc:.4f}")

# F1 and Confusion Matrix
from sklearn.metrics import f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

f1 = f1_score(pca.y_test, y_pred_test, average='macro')
print(f"Test F1 Score: {f1:.4f}")

# Compute confusion matrix (just to be sure)
cm = confusion_matrix(pca.y_test, y_pred_test)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', cbar=True, 
            xticklabels=np.unique(pca.y_test), 
            yticklabels=np.unique(pca.y_test))
plt.title("Confusion Matrix (Colored Heatmap)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

# print("Confusion Matrix:")
# df = pd.DataFrame(cm)
# print(df.to_string())



### Gaussian Mixture Model Clustering

In [ ]:
import numpy as np
from scipy.special import logsumexp
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

class GMM:
    def __init__(self, n_components, max_iters=100, tol=1e-4, reg_covar=1e-6, min_covar=1e-6):
        self.n_components = n_components  # K
        self.max_iters = max_iters
        self.tol = tol
        self.reg_covar = reg_covar  # Regularization for covariances
        self.min_covar = min_covar  # Ensure a minimum covariance threshold

    def initialize_parameters_with_kmeans(self, X):
        n_samples, n_features = X.shape
        kmeans = KMeans(n_clusters=self.n_components, random_state=42)
        kmeans.fit(X)
        
        # Initialize means with K-Means centroids
        self.means_ = kmeans.cluster_centers_

        # Initialize covariance matrices
        self.covariances_ = np.array([np.cov(X, rowvar=False) for _ in range(self.n_components)])
        # self.covariances_ = np.array([np.cov(X, rowvar=False) + self.reg_covar * np.eye(n_features)
        #                               for _ in range(self.n_components)])
        
        # Initialize weights
        self.weights_ = np.ones(self.n_components) / self.n_components

    def _ensure_positive_definite(self, cov):
        """Ensure that the covariance matrix is positive definite by adjusting eigenvalues."""
        eigvals, eigvecs = np.linalg.eigh(cov)
        eigvals = np.maximum(eigvals, self.min_covar)  # Ensure eigenvalues are above a threshold
        cov = eigvecs @ np.diag(eigvals) @ eigvecs.T
        return cov

    def compute_log_gaussian(self, X, mean, cov):
        n_features = X.shape[1]
        
        # Regularize the covariance matrix to avoid singularities
        cov += self.reg_covar * np.eye(n_features)  # Adding regularization term
        try:
            inv_cov = np.linalg.inv(cov)
            diff = X - mean
            
            # Compute log probability
            log_prob = -0.5 * (np.sum(diff @ inv_cov * diff, axis=1) +
                               np.log(np.maximum(np.linalg.det(cov), self.min_covar)) +  # Use max to avoid log(0)
                               n_features * np.log(2 * np.pi))
            return log_prob
        except np.linalg.LinAlgError:
            # If covariance is singular, return -inf for log probabilities
            return np.full(X.shape[0], -np.inf)

    def e_step(self, X):
        n_samples = X.shape[0]
        log_resp = np.zeros((n_samples, self.n_components))
        
        for k in range(self.n_components):
            log_gauss = self.compute_log_gaussian(X, self.means_[k], self.covariances_[k])
            log_resp[:, k] = np.log(self.weights_[k]) + log_gauss
        
        logsumexp_resp = logsumexp(log_resp, axis=1, keepdims=True)
        self.log_likelihood_ = np.sum(logsumexp_resp)
        
        # Avoid invalid values and ensure responsibilities sum to 1
        responsibilities = np.exp(log_resp - logsumexp_resp)
        responsibilities = np.nan_to_num(responsibilities, nan=0.0, posinf=0.0, neginf=0.0)
        return responsibilities

    def m_step(self, X, responsibilities):  # Update parameters
        n_samples, n_features = X.shape
        effective_n = responsibilities.sum(axis=0)
        
        self.weights_ = effective_n / n_samples
        self.means_ = (responsibilities.T @ X) / effective_n[:, np.newaxis]
        
        self.covariances_ = np.zeros((self.n_components, n_features, n_features))
        for k in range(self.n_components):
            diff = X - self.means_[k]
            cov_matrix = (responsibilities[:, k][:, np.newaxis] * diff).T @ diff
            cov_matrix /= effective_n[k]
            cov_matrix += self.reg_covar * np.eye(n_features)  # Regularize to avoid singularity
            self.covariances_[k] = self._ensure_positive_definite(cov_matrix)

    def fit(self, X):
        # Scale the data to have zero mean and unit variance
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        self.initialize_parameters_with_kmeans(X_scaled)  # Initialize parameters using K-Means
        prev_log_likelihood = None
        
        for iteration in range(self.max_iters):
            responsibilities = self.e_step(X_scaled)
            self.m_step(X_scaled, responsibilities)
            
            if prev_log_likelihood is not None:
                if abs(self.log_likelihood_ - prev_log_likelihood) < self.tol:
                    break
            prev_log_likelihood = self.log_likelihood_

    def predict(self, X):
        # Scale the input data before predicting
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        log_resp = np.zeros((X_scaled.shape[0], self.n_components))
        
        for k in range(self.n_components):
            log_gauss = self.compute_log_gaussian(X_scaled, self.means_[k], self.covariances_[k])
            log_resp[:, k] = np.log(self.weights_[k]) + log_gauss
        
        return np.argmax(log_resp, axis=1)


### GMM Clustering Evaluation

In [ ]:
from scipy.optimize import linear_sum_assignment
import numpy as np

def clustering_accuracy(y_true, y_pred):
    """Compute clustering accuracy using the Hungarian algorithm"""
    D = max(y_pred.max(), y_true.max()) + 1  # Number of clusters
    cost = np.zeros((D, D), dtype=int)
    
    for i in range(len(y_pred)):
        cost[y_pred[i], y_true[i]] += 1
        
    row_ind, col_ind = linear_sum_assignment(cost.max() - cost)
    total_correct = sum(cost[i, j] for i, j in zip(row_ind, col_ind))
    
    return total_correct / len(y_pred)


### Run GMM for different alpha and K

In [ ]:
results_gmm = []

for alpha in [0.8, 0.85, 0.9, 0.95]:
    # Project train data with specified alpha
    projected_train, eigvecs = pca.project(alpha,use_saved_eig=True)
    
    for K in [20, 40, 60]:
        gmm = GMM(n_components=K, max_iters=100)
        gmm.fit(projected_train)
        
        # Predict cluster labels
        y_pred_train = gmm.predict(projected_train)

        acc = clustering_accuracy(pca.y_train, y_pred_train)
        results_gmm.append((alpha, K, acc))
        print(f"Alpha: {alpha}, K: {K}, Accuracy: {acc:.4f}")
    print("--------------------------------------")


### Plot Accuracy vs K and Alpha (GMM)

In [ ]:
import matplotlib.pyplot as plt

# Accuracy vs K for each alpha
for alpha in [0.8, 0.85, 0.9, 0.95]:
    accs = [acc for a, k, acc in results_gmm if a == alpha]
    ks = [k for a, k, acc in results_gmm if a == alpha]
    plt.plot(ks, accs, marker='o', label=f'α={alpha}')

plt.xlabel('K')
plt.ylabel('Clustering Accuracy')
plt.title('Accuracy vs K for Different α (GMM)')
plt.legend()
plt.grid()
plt.show()

# Accuracy vs alpha for each K
for k in [20, 40, 60]:
    accs = [acc for a, k_val, acc in results_gmm if k_val == k]
    alphas = [a for a, k_val, acc in results_gmm if k_val == k]
    plt.plot(alphas, accs, marker='o', label=f'K={k}')

plt.xlabel('α')
plt.ylabel('Clustering Accuracy')
plt.title('Accuracy vs α for Different K (GMM)')
plt.legend()
plt.grid()
plt.show()


### Evaluate Best GMM Model on the Test Set

In [ ]:
# Find best (alpha, K) combination
best_alpha, best_K, best_acc = max(results_gmm, key=lambda x: x[2])

print(f"Best alpha: {best_alpha}, Best K: {best_K}, Accuracy: {best_acc:.4f}")

# Project training and test data with best_alpha
projected_train, eigvecs = pca.project(best_alpha,use_saved_eig=True)

mean_face = np.mean(pca.D_train, axis=0)
D_test_centered = pca.D_test - mean_face
projected_test = D_test_centered @ eigvecs

# Fit GMM on train projection
gmm_best = GMM(n_components=best_K, max_iters=100)
gmm_best.fit(projected_train)

# Predict on test set
y_pred_test = gmm_best.predict(projected_test)

# Evaluate clustering accuracy on test set
test_acc = clustering_accuracy(pca.y_test, y_pred_test)
print(f"Test Accuracy: {test_acc:.4f}")

# F1 and Confusion Matrix for GMM
from sklearn.metrics import f1_score, confusion_matrix
import seaborn as sns

# Compute F1 score
f1 = f1_score(pca.y_test, y_pred_test, average='macro')
print(f"Test F1 Score: {f1:.4f}")

# Compute Confusion Matrix
cm = confusion_matrix(pca.y_test, y_pred_test)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', cbar=True, 
            xticklabels=np.unique(pca.y_test), 
            yticklabels=np.unique(pca.y_test))
plt.title("Confusion Matrix (GMM on Test Set)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()


### Bonus (Autoencoders)

In [ ]:
import numpy as np

class Autoencoder:
    def __init__(self, input_dim, hidden_dim):
        # Xavier initialization chooses initial weights smartly so they are not too small or too big
        # Used in encoding 
        limit1 = np.sqrt(6 / (input_dim + hidden_dim))
        self.W1 = np.random.uniform(-limit1, limit1, (hidden_dim, input_dim))   
        self.b1 = np.zeros((hidden_dim,))

        # Used in decoding 
        limit2 = np.sqrt(6 / (hidden_dim + input_dim))
        self.W2 = np.random.uniform(-limit2, limit2, (input_dim, hidden_dim))
        self.b2 = np.zeros((input_dim,))

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def sigmoid_deriv(self, x):
        s = self.sigmoid(x)
        return s * (1 - s)

    def encode(self, X):
        self.Z1 = X @ self.W1.T + self.b1  # Pre-activation
        self.A1 = self.sigmoid(self.Z1)    # Activation
        return self.A1

    def decode(self, A1):
        self.Z2 = A1 @ self.W2.T + self.b2
        self.A2 = self.sigmoid(self.Z2)
        return self.A2

    def forward(self, X):
        return self.decode(self.encode(X))

    def compute_loss(self, X, X_hat):
        return np.mean((X - X_hat) ** 2)

    def train(self, X, lr, epochs):
        losses = []
        patience = 10  # Number of epochs to wait for improvement
        best_loss = float('inf')
        no_improvement_count = 0

        for epoch in range(epochs):
            # Forward pass
            A1 = self.encode(X)
            A2 = self.decode(A1)
            loss = self.compute_loss(X, A2)

            # Early stopping check
            if loss < best_loss:
                best_loss = loss
                no_improvement_count = 0
            else:
                no_improvement_count += 1

            if no_improvement_count >= patience:
                print(f"Early stopping at epoch {epoch}, best loss: {best_loss:.5f}")
                break

            # Backpropagation
            dA2 = 2 * (A2 - X) / X.shape[0]  # MSE derivative
            dZ2 = dA2 * self.sigmoid_deriv(self.Z2)     # dL/dZ2
            dW2 = dZ2.T @ A1        # dL/dW2
            db2 = np.sum(dZ2, axis=0)       # dL/db2

            dA1 = dZ2 @ self.W2     # dL/dA1
            dZ1 = dA1 * self.sigmoid_deriv(self.Z1)     # dL/dZ1
            dW1 = dZ1.T @ X     # dL/dW1
            db1 = np.sum(dZ1, axis=0)     # dL/db1

            # Update weights
            self.W1 -= lr * dW1
            self.b1 -= lr * db1
            self.W2 -= lr * dW2
            self.b2 -= lr * db2

            losses.append(loss)
            if epoch % 20 == 0:
                print(f"Epoch {epoch}, Loss: {loss:.5f}, lr: {lr}")

        plt.plot(losses)
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Training Loss Curve")
        plt.show()
        return losses
    
    
    def visualize_reconstruction(self, X_original, index, image_shape=(112, 92)):
        
        original = X_original[index]
        # Reshape before encoding to make it [1, 10304] instead of [10304,]
        encoded = self.encode(original.reshape(1, -1))
        # Returns it to 1D [10304, ] instead of [1, 10304] after decoding
        decoded = self.decode(encoded).reshape(-1)

        fig, axs = plt.subplots(1, 2, figsize=(8, 4))

        axs[0].imshow(original.reshape(image_shape), cmap='gray')
        axs[0].set_title("Original")
        axs[0].axis('off')

        axs[1].imshow(decoded.reshape(image_shape), cmap='gray')
        axs[1].set_title("Reconstruction")
        axs[1].axis('off')

        plt.suptitle(f"Autoencoder Reconstruction (sample #{index})")
        plt.show()




### Autoencoder Training

In [ ]:
# Normalize
D_train_norm = D_train / 255.0
D_test_norm = D_test / 255.0


# Initialize
print("Trying with hidden_dim=4000, lr=0.05, epochs=250")
auto1 = Autoencoder(input_dim=10304, hidden_dim=4000)
# Train
auto1.train(D_train_norm, lr=0.05, epochs=250)
# Visualize
auto1.visualize_reconstruction(D_train_norm, index=22)
auto1.visualize_reconstruction(D_train_norm, index=5)
auto1.visualize_reconstruction(D_train_norm, index=150)

auto1.visualize_reconstruction(D_test_norm, index=20)
auto1.visualize_reconstruction(D_test_norm, index=150)
auto1.visualize_reconstruction(D_test_norm, index=10)

print("Trying with hidden_dim=500, lr=0.1, epochs=100")
auto2 = Autoencoder(input_dim=10304, hidden_dim=500)
# Train
auto2.train(D_train_norm, lr=0.1, epochs=100)
# Visualize
auto2.visualize_reconstruction(D_train_norm, index=22)
auto2.visualize_reconstruction(D_train_norm, index=5)
auto2.visualize_reconstruction(D_train_norm, index=150)

auto2.visualize_reconstruction(D_test_norm, index=30)
auto2.visualize_reconstruction(D_test_norm, index=100)
auto2.visualize_reconstruction(D_test_norm, index=110)


print("Trying with hidden_dim=1000, lr=0.01, epochs=150")
auto3 = Autoencoder(input_dim=10304, hidden_dim=1000)
# Train
auto3.train(D_train_norm, lr=0.01, epochs=150)
# Visualize
auto3.visualize_reconstruction(D_train_norm, index=20)
auto3.visualize_reconstruction(D_train_norm, index=10)
auto3.visualize_reconstruction(D_train_norm, index=150)

auto3.visualize_reconstruction(D_test_norm, index=30)
auto3.visualize_reconstruction(D_test_norm, index=100)
auto3.visualize_reconstruction(D_test_norm, index=110)




# Train Autoencoder with different hyperparameters
# hidden_dims = [50, 100, 200]
# learning_rates = [0.01, 0.05, 0.1]
# epochs_list = [50, 100, 200]  # Different number of epochs to try
# 
# for hidden_dim in hidden_dims:
#     for lr in learning_rates:
#         for epochs in epochs_list:
#             print(f"\nTraining Autoencoder: hidden_dim={hidden_dim}, lr={lr}, epochs={epochs}")
# 
#             # Initialize
#             auto = Autoencoder(input_dim=10304, hidden_dim=hidden_dim)
#             # Train
#             auto.train(D_train_norm, lr=lr, epochs=epochs)
#             # Visualize
#             auto.visualize_reconstruction(D_train_norm, index=22)
#             auto.visualize_reconstruction(D_train_norm, index=5)
#             auto.visualize_reconstruction(D_train_norm, index=150)
# 
#             encoded_features = auto.encode(D_train_norm)

### K-Means on Encoded Data for Autoencoders

In [ ]:
# KMeans clustering on encoded data
results_kmeans_autoencoder = []
for dim in [500, 1000]:
    for lr in [0.05, 0.1]:
        for epochs in [250]:
            for K in [20, 40, 60]:
                print(f"\nTraining Autoencoder: hidden_dim={dim}, lr={lr}, epochs={epochs}, K={K}")

                # Initialize
                auto = Autoencoder(input_dim=10304, hidden_dim=dim)
                # Train
                auto.train(D_train_norm, lr=lr, epochs=epochs)
                
                # Encode the data
                D_train_encoded = auto.encode(D_train_norm)
                D_test_encoded = auto.encode(D_test_norm)
                
                # resize
                desired_dim = 100
                D_train_resized = D_train_encoded[:, :desired_dim]    
                
                # Fit KMeans
                kmeans = CustomKMeans(n_clusters=K)
                kmeans.fit(D_train_resized)

                acc = clustering_accuracy(y_train, kmeans.labels_)
                results_kmeans_autoencoder.append((dim, lr, epochs, K, acc))
                print(f"Hidden Dim: {dim}, Learning Rate: {lr}, Epochs: {epochs}, K: {K}, Accuracy: {acc:.4f}")
                print("------------------------------------------------------------------------------------------")

### Plot Accuracy vs K for Autoencoder (K-Means)

In [ ]:
import matplotlib.pyplot as plt

for dim in [500, 1000]:
    for lr in [0.05, 0.1]:
        accs = [acc for d, l, e, k, acc in results_kmeans_autoencoder if d == dim and l == lr]
        ks = [k for d, l, e, k, acc in results_kmeans_autoencoder if d == dim and l == lr]
        plt.plot(ks, accs, marker='o', label=f'Dim={dim}, lr={lr}')
        
    
plt.xlabel('K')
plt.ylabel('Clustering Accuracy')
plt.title('Accuracy vs K for Autoencoder + KMeans')
plt.legend()
plt.grid()
plt.show()

### Evaluate Best K-Means on Test Set for Autoencoder

In [ ]:
best_dim, best_lr, best_epochs, best_K, best_acc = max(results_kmeans_autoencoder, key=lambda x: x[4])
print(f"Best K: {best_K}, Accuracy: {best_acc:.4f}, Hidden Dim: {best_dim}, Learning Rate: {best_lr}, Epochs: {best_epochs}")

# Initialize the Autoencoder with the best parameters
auto = Autoencoder(input_dim=10304, hidden_dim=best_dim)
auto.train(D_train_norm, lr=best_lr, epochs=best_epochs)

D_train_encoded = auto.encode(D_train_norm)
D_test_encoded = auto.encode(D_test_norm)

desired_dim = 100
D_train_resized = D_train_encoded[:, :desired_dim]
D_test_resized = D_test_encoded[:, :desired_dim]

kmeans_best = CustomKMeans(n_clusters=best_K)
kmeans_best.fit(D_train_resized)

# Predict on the resized encoded test set
y_pred_test = kmeans_best.predict(D_test_resized)

# Evaluate the clustering accuracy
acc = clustering_accuracy(y_test, y_pred_test)
print(f"Test Accuracy: {acc:.4f}")

# F1 Score
f1 = f1_score(y_test, y_pred_test, average='macro')
print(f"Test F1 Score: {f1:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', cbar=True)
plt.title('Confusion Matrix for Autoencoder + KMeans')
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()


### GMM on Encoded Data for Autoencoders

In [ ]:
# Encode the data
D_train_encoded = auto.encode(D_train_norm)
D_test_encoded = auto.encode(D_test_norm)

desired_dim = 100 
D_train_resized = D_train_encoded[:, :desired_dim] 
D_test_resized = D_test_encoded[:, :desired_dim] 

# GMM clustering on encoded data
results_gmm_autoencoder = []
for dim in [500, 1000]:
    for lr in [0.05, 0.1]:
        for epochs in [250]:
            for K in [20, 40, 60]:
                print(f"\nTraining Autoencoder: hidden_dim={dim}, lr={lr}, epochs={epochs}, K={K}")

                # Initialize
                auto = Autoencoder(input_dim=10304, hidden_dim=dim)
                # Train
                auto.train(D_train_norm, lr=lr, epochs=epochs)
                # Encode the data
                D_train_encoded = auto.encode(D_train_norm)
                D_test_encoded = auto.encode(D_test_norm)
                
                desired_dim = 100
                D_train_resized = D_train_encoded[:, :desired_dim]    

                # Fit GMM
                gmm = GMM(n_components=K)
                gmm.fit(D_train_encoded)
                y_pred = gmm.predict(D_train_encoded)
                acc = clustering_accuracy(y_train, y_pred)
                results_gmm_autoencoder.append((dim, lr, epochs, K, acc))
                print(f"Hidden Dim: {dim}, Learning Rate: {lr}, Epochs: {epochs}, K: {K}, Accuracy: {acc:.4f}")
                print("------------------------------------------------------------------------------------------")

### Plot Accuracy vs K for Autoencoder (GMM)

In [ ]:
import matplotlib.pyplot as plt

for dim in [500, 1000]:
    for lr in [0.05, 0.1]:
        accs = [acc for d, l, e, k, acc in results_gmm_autoencoder if d == dim and l == lr]
        ks = [k for d, l, e, k, acc in results_gmm_autoencoder if d == dim and l == lr]
        plt.plot(ks, accs, marker='o', label=f'Dim={dim}, lr={lr}')
        
    
plt.xlabel('K')
plt.ylabel('Clustering Accuracy')
plt.title('Accuracy vs K for Autoencoder + KMeans')
plt.legend()
plt.grid()
plt.show()

### Evaluate Best GMM on Test Set for Autoencoder

In [ ]:
best_dim, best_lr, best_dim, best_epochs, best_acc = max(results_gmm_autoencoder, key=lambda x: x[4])
print(f"Best K: {best_K}, Accuracy: {best_acc:.4f}, Hidden Dim: {best_dim}, Learning Rate: {best_lr}, Epochs: {best_epochs}")

auto = Autoencoder(input_dim=10304, hidden_dim=best_dim)
auto.train(D_train_norm, lr=best_lr, epochs=best_epochs)

# Encode the training and test data
D_train_encoded = auto.encode(D_train_norm)
D_test_encoded = auto.encode(D_test_norm)

desired_dim = 100 
D_train_resized = D_train_encoded[:, :desired_dim]
D_test_resized = D_test_encoded[:, :desired_dim]

# Train best GMM again
gmm_best = GMM(n_components=best_K)
gmm_best.fit(D_train_encoded)

# Predict on encoded test set
y_pred_test = gmm_best.predict(D_test_encoded)

# Evaluate
acc = clustering_accuracy(y_test, y_pred_test)
print(f"Test Accuracy: {acc:.4f}")

f1 = f1_score(y_test, y_pred_test, average='macro')
print(f"Test F1 Score: {f1:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', cbar=True)
plt.title('Confusion Matrix for Autoencoder + GMM')
plt.show()
